In [1]:
!pip install psycopg2-binary


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import RealDictCursor


# Establish the connection
conn = psycopg2.connect(
    host="localhost",
    user="postgres",
    port= 5432, # Default user for Postgres is usually 'postgres', not 'root'
    password="Yatin0987@#",
    dbname="postgres"  # Parameter is usually 'dbname' in psycopg2
)





In [3]:
import psycopg2

def connect_to_db():
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        user="postgres",
        password="Yatin0987@#",
        dbname="postgres"
    )
    return conn




In [4]:
conn = connect_to_db()
print("Connected successfully")


Connected successfully


In [5]:
from psycopg2.extras import RealDictCursor

db = connect_to_db()
cursor = db.cursor(cursor_factory=RealDictCursor)


In [6]:
cursor

<cursor object at 0x0000021C4C35C6E0; closed: 0>

In [7]:
'select count(*) as total_suppliers from "suppliers"'

'select count(*) as total_suppliers from "suppliers"'

In [8]:
cursor.execute('select count(*) as total_suppliers from "suppliers"')

In [9]:
row = cursor.fetchone()

In [10]:
list(row.values())[0]

50

In [11]:
queries = {
    "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

    "Total Products": "SELECT COUNT(*) AS count FROM products",

    "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

    "Total Sale Value (Last 3 Months)": """
        SELECT ROUND(SUM(ABS(se.change_quantity) * p.price)::numeric, 2) AS total_sale
        FROM stock_entries se
        JOIN products p ON se.product_id = p.product_id
        WHERE se.change_type = 'Sale'
        AND se.entry_date >= (
            SELECT MAX(entry_date) - INTERVAL '3 months'
            FROM stock_entries
        )
    """,

    "Total Restock Value (Last 3 Months)": """
        SELECT ROUND(SUM(se.change_quantity * p.price)::numeric, 2) AS total_restock
        FROM stock_entries se
        JOIN products p ON se.product_id = p.product_id
        WHERE se.change_type = 'Restock'
        AND se.entry_date >= (
            SELECT MAX(entry_date) - INTERVAL '3 months'
            FROM stock_entries
        )
    """,

    "Below Reorder & No Pending Reorders": """
        SELECT COUNT(*) AS below_reorder
        FROM products p
        WHERE p.stock_quantity < p.reorder_level
        AND p.product_id NOT IN (
            SELECT DISTINCT product_id
            FROM reorders
            WHERE status = 'Pending'
        )
    """
}

In [12]:
result = {}

for label, query in queries.items():
    cursor.execute(query)
    row = cursor.fetchone()
    result[label] = list(row.values())[0]


In [13]:
result

{'Total Suppliers': 50,
 'Total Products': 206,
 'Total Categories Dealing': 5,
 'Total Sale Value (Last 3 Months)': None,
 'Total Restock Value (Last 3 Months)': Decimal('59994.00'),
 'Below Reorder & No Pending Reorders': 14}

In [14]:
def get_basic_info(cursor):
    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",
    
        "Total Products": "SELECT COUNT(*) AS count FROM products",
    
        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",
    
        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price)::numeric, 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
            AND se.entry_date >= (
                SELECT MAX(entry_date) - INTERVAL '3 months'
                FROM stock_entries
            )
        """,
    
        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price)::numeric, 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
            AND se.entry_date >= (
                SELECT MAX(entry_date) - INTERVAL '3 months'
                FROM stock_entries
            )
        """,
    
        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
            AND p.product_id NOT IN (
                SELECT DISTINCT product_id
                FROM reorders
                WHERE status = 'Pending'
            )
        """
            }
    result = {}
    
    for label, query in queries.items():
        cursor.execute(label)
        row = cursor.fetchone()
        result[label] = list(row.values())[0]

    return result

    

In [15]:
queries = {
            "Suppliers Contact Details": """
                SELECT supplier_name, contact_name, email, phone
                FROM suppliers;
            """,
    
            "Products with Supplier and Stock": """
                SELECT 
                    p.product_name,
                    s.supplier_name,
                    p.stock_quantity,
                    p.reorder_level
                FROM products p
                JOIN suppliers s 
                    ON p.supplier_id = s.supplier_id
                ORDER BY p.product_name ASC;
            """,
    
            "Products Needing Reorder": """
                SELECT 
                    product_name,
                    stock_quantity,
                    reorder_level
                FROM products
                WHERE stock_quantity <= reorder_level;
            """
        }

tables = {}

for label, query in queries.items():
    cursor.execute(query)
    tables[label] = cursor.fetchall()



In [16]:
 tables

{'Suppliers Contact Details': [RealDictRow([('supplier_name',
                'Anderson-Thompson'),
               ('contact_name', 'Bonnie Davis'),
               ('email', 'zacharysanchez@hotmail.com'),
               ('phone', '829.485.9853x0522')]),
  RealDictRow([('supplier_name', 'Rowland Ltd'),
               ('contact_name', 'Beth Stevens'),
               ('email', 'joshua60@yahoo.com'),
               ('phone', '(779)942-0726')]),
  RealDictRow([('supplier_name', 'Baxter-Meadows'),
               ('contact_name', 'Lisa Lewis'),
               ('email', 'andersonchristina@yahoo.com'),
               ('phone', '449-766-7325')]),
  RealDictRow([('supplier_name', 'Wilson, Graham and Williams'),
               ('contact_name', 'David Martinez'),
               ('email', 'abrown@hotmail.com'),
               ('phone', '+1-653-827-5215x266')]),
  RealDictRow([('supplier_name', 'Smith, Kennedy and Moreno'),
               ('contact_name', 'Victoria Gonzalez'),
               ('email'

In [17]:
def get_additional_tables(cursor):
    queries = {
            "Suppliers Contact Details": """
                SELECT supplier_name, contact_name, email, phone
                FROM suppliers;
            """,
    
            "Products with Supplier and Stock": """
                SELECT 
                    p.product_name,
                    s.supplier_name,
                    p.stock_quantity,
                    p.reorder_level
                FROM products p
                JOIN suppliers s 
                    ON p.supplier_id = s.supplier_id
                ORDER BY p.product_name ASC;
            """,
    
            "Products Needing Reorder": """
                SELECT 
                    product_name,
                    stock_quantity,
                    reorder_level
                FROM products
                WHERE stock_quantity <= reorder_level;
            """
        }

    tables = {}
    
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

    return tables

    

In [18]:
get_additional_tables(cursor)

{'Suppliers Contact Details': [RealDictRow([('supplier_name',
                'Anderson-Thompson'),
               ('contact_name', 'Bonnie Davis'),
               ('email', 'zacharysanchez@hotmail.com'),
               ('phone', '829.485.9853x0522')]),
  RealDictRow([('supplier_name', 'Rowland Ltd'),
               ('contact_name', 'Beth Stevens'),
               ('email', 'joshua60@yahoo.com'),
               ('phone', '(779)942-0726')]),
  RealDictRow([('supplier_name', 'Baxter-Meadows'),
               ('contact_name', 'Lisa Lewis'),
               ('email', 'andersonchristina@yahoo.com'),
               ('phone', '449-766-7325')]),
  RealDictRow([('supplier_name', 'Wilson, Graham and Williams'),
               ('contact_name', 'David Martinez'),
               ('email', 'abrown@hotmail.com'),
               ('phone', '+1-653-827-5215x266')]),
  RealDictRow([('supplier_name', 'Smith, Kennedy and Moreno'),
               ('contact_name', 'Victoria Gonzalez'),
               ('email'

In [19]:
def add_new_manual_id(cursor, db, p_name, p_category, p_price, p_stock, p_reorder, p_supplier):
    proc_call = "CALL AddNewProductManualID(%s, %s, %s, %s, %s, %s)"
    params = (p_name, p_category, p_price, p_stock, p_reorder, p_supplier)

    cursor.execute(proc_call, params)
    db.commit()   # ✅ parentheses = actually commits


In [20]:
def get_categories(cursor):
    cursor.execute("SELECT DISTINCT category FROM products ORDER BY category ASC")
    rows = cursor.fetchall()
    return [row["category"] for row in rows]

In [21]:
get_categories(cursor)

['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']

In [22]:
def get_suppliers(cursor):
    cursor.execute("SELECT supplier_id, supplier_name FROM suppliers ORDER BY supplier_name ASC")
    return cursor.fetchall()



In [23]:
get_suppliers(cursor)


[RealDictRow([('supplier_id', 1), ('supplier_name', 'Anderson-Thompson')]),
 RealDictRow([('supplier_id', 48), ('supplier_name', 'Armstrong-Vance')]),
 RealDictRow([('supplier_id', 24),
              ('supplier_name', 'Barker, White and Carson')]),
 RealDictRow([('supplier_id', 25), ('supplier_name', 'Barrett Ltd')]),
 RealDictRow([('supplier_id', 3), ('supplier_name', 'Baxter-Meadows')]),
 RealDictRow([('supplier_id', 19), ('supplier_name', 'Charles Inc')]),
 RealDictRow([('supplier_id', 41), ('supplier_name', 'Clark Group')]),
 RealDictRow([('supplier_id', 23), ('supplier_name', 'Douglas Ltd')]),
 RealDictRow([('supplier_id', 32), ('supplier_name', 'Elliott-Ayers')]),
 RealDictRow([('supplier_id', 7), ('supplier_name', 'Evans Inc')]),
 RealDictRow([('supplier_id', 27),
              ('supplier_name', 'Franklin, Kane and Price')]),
 RealDictRow([('supplier_id', 15), ('supplier_name', 'Freeman-Gordon')]),
 RealDictRow([('supplier_id', 13), ('supplier_name', 'Gallagher-Miller')]),
 Real

In [24]:
def get_pending_reorders(cursor):
    cursor.execute("""
        SELECT r.reorder_id, p.product_name, r.order_quantity, r.status
        FROM reorders r
        JOIN products p ON r.product_id = p.product_id
        WHERE r.status = 'Pending'
    """)
    return cursor.fetchall()
